# Agent liniowy — przepływ krok po kroku

Ten notebook uruchamia po kolei umiejętności z `src/skills/*` i pokazuje wynik
**każdego etapu** — dokładnie to, co robi `agent_linear/pipeline.py`, ale
interaktywnie, żeby zobaczyć outputy w przepływie.

> To **nie** jest baseline (naiwny jeden wielki prompt) — to agent liniowy, który
> spina umiejętności w sekwencję. Pierwsza komórka ustawia katalog i model.

In [3]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

# Model: lokalnie przez Ollamę (qwen). Dla OpenAI wpisz własny klucz/model.
os.environ["LLM_BASE_URL"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"]  = "ollama"
os.environ["LLM_MODEL"]    = "qwen2.5:14b"

from src.loader import load_all
from src.sources import prepare_input_texts
from src.skills.file_description.main import generate_file_description
from src.skills.tasks.main import generate_tasks
from src.skills.make_task.main import make_task
from src.skills.strategy.main import generate_strategy
from src.skills.document.main import generate_document

GOAL = "Wygeneruj apelację z perspektywy obrony"
GOAL

'Wygeneruj apelację z perspektywy obrony'

## 0. Wczytanie akt

In [4]:
documents = load_all()
print(f"Wczytano {len(documents)} dokumentów")
for d in documents:
    print(" -", d.filename)

Wczytano 16 dokumentów
 - 02_Notatka_urzędowa_z_interwencji_dotyczącej_nietrzeźwego_kierowcy.pdf
 - 03_Protokół_z_badania_stanu_trzeźwości_analizatorem_wydechu.pdf
 - 04_Postanowienie_o_wszczęciu_dochodzenia.pdf
 - 05_Protokół_przesłuchania_świadka.pdf
 - 06_Protokół_przesłuchania_świadka.pdf
 - 07_Protokół_przesłuchania_świadka.pdf
 - 08_Postanowienie_o_przedstawieniu_zarzutów.pdf
 - 09_Protokół_przesłuchania_podejrzanego.pdf
 - 10_Protokół_przesłuchania_podejrzanego.pdf
 - 11_Akt_oskarżenia.pdf
 - 12_Protokół_rozprawy_głównej.pdf
 - 13_Odpowiedź_na_wezwanie_Sądu_Rejonowego_dotyczące_zarządzania_cmentarzem.pdf
 - 14_Opinia_biegłego_sądowego_z_zakresu_toksykologii_sądowej.pdf
 - 15_Protokół_rozprawy_głównej.pdf
 - 16_Wyrok.pdf
 - 17_Wniosek_o_sporządzenie_uzasadnienia.pdf


In [6]:
documents[0]

Document(filename='02_Notatka_urzędowa_z_interwencji_dotyczącej_nietrzeźwego_kierowcy.pdf', text='Komisariat Policji w Balicach Balice, 31 grudnia 2024 r.\nsierż. Wiesław Wilk\nNOTATKA URZĘDOWA\nW dniu datowania pełniłem służbę w obchodzie z post. Sylwią Sikorą na terenie\nKP Balice.\nOk. godz. 10.40 z polecenia dyżurnego KP Balice udaliśmy się do miejscowości\nSzczyglice w rejon cmentarza, gdzie według zgłoszenia kierujący pojazdem marki Jeep\nWrangler może być nietrzeźwy. Na miejscu zastano zgłaszającego: Karola Kota (Kot), który\noświadczył, że widział, jak kierujący pojazdem na terenie należącym do cmentarza usiłował\nzawrócić pojazd w alejce. Zgłaszający stwierdził, że kierowca prawdopodobnie jest\nnietrzeźwy. Następnie Karol Kot wskazał nam pojazd i udał się na pogrzeb. W pojeździe\nwskazanym przez Karola Kota na miejscu kierowcy zastano mężczyznę, od którego\nwyczuwalna była silna woń alkoholu. Wyciągnięto kluczyki ze stacyjki. Mężczyzną okazał\nsię być:\nDaniel Dzik, s. Jana i 

## 1. `generate_file_description` — opis każdego dokumentu

Surowe akta → zwięzłe, ukierunkowane na cel opisy ("mapa" sprawy).

In [ ]:
described = [generate_file_description(doc, GOAL) for doc in documents]
for d in described:
    print(f"### {d.name}")
    print(d.title)
    print(d.description)
    print()

## 2. `generate_tasks` — plan analizy

Każdy krok wskazuje, **które** dokumenty są mu potrzebne (`input`).

In [ ]:
tasks = generate_tasks(GOAL, described)
for i, t in enumerate(tasks.thoughts, 1):
    print(f"[{i}] {t.action}")
    print(f"    dokumenty: {t.input}")
    print(f"    rozumowanie: {t.reasoning}")
    print()

## 3. `make_task` — wykonanie kroków na wybranych dokumentach

Sedno: `prepare_input_texts` daje krokowi **tylko** wskazane dokumenty (selektywny kontekst).

In [ ]:
task_outputs = []
for i, task in enumerate(tasks.thoughts, 1):
    sources_text = prepare_input_texts(documents, task.input)
    out = make_task(GOAL, task, sources_text)
    task_outputs.append(out)
    print(f"[{i}] {out.action}")
    print(out.output)
    print("-" * 70)

## 4. `generate_strategy` — wybór strategii

In [ ]:
strategy = generate_strategy(GOAL, task_outputs)
print("Definicja sukcesu:\n", strategy.definition_of_success)
print("\nMożliwe strategie:\n", strategy.strategies)
print("\nNajlepsza strategia:\n", strategy.best_strategy)

## 5. `generate_document` — gotowa apelacja

In [ ]:
document = generate_document(GOAL, strategy, task_outputs)
print(document.tekst)

## 6. Ewaluacja

Pokrywalność wygenerowanej apelacji względem `data/eval.json`.

In [ ]:
from src.eval.coverage import evaluate

result = evaluate(document.tekst)
print(f"Pokryte: {result.covered}/{result.total} = {result.score:.0%}\n")
for r in result.results:
    print(("✅" if r.covered else "❌"), r.issue[:80])